# Notebook 2: Fine-tune CHURRO and score it

This trains a **LoRA adapter** for **CHURRO-3B**, a handwriting model trained on 100,000
historical pages, on whole pages from all three collections at once. Then it scores the result on
the team's 55 dev pages with the competition's own `metric.py`, per collection.

It's the recipe from the SageMaker lesson (which took Kade from a CER of 0.267 to 0.065), with
the team's dev pages as the test, survey spreads read one page at a time, and cleaned labels.

**Use an `ml.g5.xlarge` (or `ml.g6.xlarge`) notebook instance.** On a T4 (`ml.g4dn`) it is two to
three times slower, and training in fp16 there can fail with a `nan` loss.
Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Install the libraries

Run this after every start of the instance, and before any other cell.

In [ ]:
%pip install -q -r requirements.txt

import importlib.metadata
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

# The model also needs torchvision in exactly the version and build made for this kernel's
# PyTorch (2.8.0 goes with 0.23.0, 2.9.1 with 0.24.1, ...). conda_pytorch kernels have it.
def torchvision_works():
    return subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True).returncode == 0

torch_version, _, build = importlib.metadata.version("torch").partition("+")
if not torchvision_works():
    _, minor, patch = (torch_version.split(".") + ["0"])[:3]
    wanted = f"torchvision==0.{int(minor) + 15}.{patch}"
    sources = ([["--index-url", f"https://download.pytorch.org/whl/{build}"]] if build else []) + [[]]
    for source in sources:
        print(f"Installing {wanted} for PyTorch {torch_version}{'+' + build if build else ''} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wanted, *source])
        if torchvision_works():
            break
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchvision"])
    else:
        raise SystemExit("Couldn't install a torchvision that works with this kernel's PyTorch. Switch the "
                         "kernel to conda_pytorch_p310 (Kernel → Change Kernel) and run this cell again.")
print(f"PyTorch {importlib.metadata.version('torch')}, torchvision {importlib.metadata.version('torchvision')}: ready.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here
print("Working folder:", WORK)

## Step 3. Check the GPU, the disk and the data

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise SystemExit("This instance has no GPU. Stop it and change its type to ml.g5.xlarge (README.md, Part 1).")
gpu = torch.cuda.get_device_properties(0)
bf16 = torch.cuda.get_device_capability()[0] >= 8
print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, {'bf16' if bf16 else 'fp16 (no bf16 on this GPU)'}")
if not bf16:
    print("WARNING: this GPU (a T4?) trains in fp16, which can give a nan loss with this model, and is 2-3x slower.\n"
          "  An ml.g5.xlarge or ml.g6.xlarge instance is the safer choice (README.md, Part 1).")
free = shutil.disk_usage(WORK).free / 1e9
print(f"Disk: {free:.0f} GB free" + ("  WARNING: under 15 GB; make the volume bigger" if free < 15 else ""))
missing = [n for n in ("train_items.csv", "dev_pages.csv", "parts.csv") if not os.path.exists(os.path.join(DATA, n))]
if missing:
    raise SystemExit(f"Missing {', '.join(missing)}. Run Notebook 1 first.")
import pandas as pd
items = pd.read_csv(os.path.join(DATA, "train_items.csv"), keep_default_na=False)
print("Training images:", ", ".join(f"{c} {n}" for c, n in items.category.value_counts().items()))

## Step 4. Choose the run and the settings

| `RUN` | What it does | About how long on `ml.g5.xlarge` |
|---|---|---|
| `"smoke"` | 12 images, 1 epoch, 3 dev pages. Only proves everything works. | 10–15 min the first time (mostly downloading the 7.5 GB model, once) |
| `"baseline"` | No training: scores all 55 dev pages with CHURRO as it is. The number to beat. | 15–30 min |
| `"full"` | Trains with the settings below, then scores all 55 dev pages. | 1 to 1.5 hours |

On an `ml.g4dn.xlarge`, multiply by about 2.5. README.md explains each setting in plain words.
**Change one at a time**, and write each run's settings and scores in the team's results table.

Each combination of settings gets its own folder in `runs/`. Running the same combination again
continues it from its last saved epoch; to start over, delete its folder.

In [ ]:
RUN = "smoke"   # "smoke", then "baseline", then "full"

settings = {
    "epochs": 2,                    # passes through all the training images
    "lr": 2e-4,                     # learning rate: how big each correction is
    "lora_r": 16,                   # adapter size
    "lora_alpha": 32,               # adapter strength; keep at 2 x lora_r
    "grad_accum": 8,                # images per correction
    "augment": "dominy_accounts",   # collections whose training images get small random changes ("" = none, "all")
    "train_vision": 0,              # 1 = also train the image side of the model
    "max_pixels": 1280 * 28 * 28,   # image detail: about a megapixel
    "curve_pages": 3,               # dev pages per collection scored after every epoch, for the curve
}
if RUN == "smoke":
    settings.update(epochs=1, max_train_items=12, dev_pages=3, curve_pages=0)
elif RUN == "baseline":
    settings = {"epochs": 0, "max_pixels": settings["max_pixels"]}

short = {"epochs": "ep", "lr": "lr", "lora_r": "r", "lora_alpha": "a", "grad_accum": "ga", "augment": "aug",
         "train_vision": "vis", "max_pixels": "px"}
run_name = RUN if RUN != "full" else "full-" + "-".join(
    f"{short[k]}{str(settings[k]).replace('_accounts', '').replace(',', '+') or 'none'}" for k in short)
run_dir = os.path.join(RUNS, run_name)
os.makedirs(run_dir, exist_ok=True)
print("This run's folder:", run_dir)

## Step 5. Train and score

The progress appears below and is saved to `train.log` in the run's folder. If the browser
disconnects, it keeps going while the instance runs; open `train.log` to see where it is. To stop
it, use Kernel → Interrupt Kernel.

What to look for: `[dev before training]` and `[dev after epoch N]` (a few dev pages, for the
curve), `loss` going down, then `[dev after training]` with the full dev score per collection.

In [ ]:
args = {**settings, "data_dir": DATA, "out_dir": run_dir}
flags = " ".join(f"--{k} {v}" for k, v in args.items() if v != "")
log = os.path.join(run_dir, "train.log")
python = sys.executable
!{python} {SCRIPTS}/finetune_churro.py {flags} 2>&1 | tee -a {log}

## Step 6. The scores

CER is the share of characters wrong (lower is better). Kaggle's score is the **mean of the three
collections**, each counting for a third. If you've run `"baseline"`, the table compares this run
with it.

In [ ]:
import json

summary = json.load(open(os.path.join(run_dir, "summary.json")))
table = pd.DataFrame({"this run": summary["dev_cer"]})
base = os.path.join(RUNS, "baseline", "summary.json")
if os.path.exists(base) and run_name != "baseline":
    table["baseline (no training)"] = pd.Series(json.load(open(base))["dev_cer"])
    table["improvement"] = table["baseline (no training)"] - table["this run"]
print(table.round(3).to_string())
if "train_minutes" in summary:
    print(f"\nTraining took {summary['train_minutes']} min on {summary['train_images']} images.")

curve = json.load(open(os.path.join(run_dir, "curve.json")))
if curve["dev_cer_by_epoch"]:
    print("\nA few dev pages per collection after each epoch (0 = before training):")
    print(pd.DataFrame(curve["dev_cer_by_epoch"]).round(3).to_string(index=False))
if curve["loss"]:
    import matplotlib.pyplot as plt
    loss = pd.DataFrame(curve["loss"])
    plt.figure(figsize=(6, 3)); plt.plot(loss.step, loss.loss); plt.xlabel("step"); plt.ylabel("training loss")
    plt.tight_layout(); plt.show()

## Step 7. Look at one page

The page that scored worst first; type any dev page name to see another. Green is what the
model wrote that isn't in the label; red, struck through, is what it missed or got wrong.

In [ ]:
import difflib
import html
from IPython.display import HTML, display

def coloured_diff(reference, prediction):
    """Green: what the model wrote that isn't in the label. Red, struck through: what it missed or got wrong."""
    out = []
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(None, reference, prediction, autojunk=False).get_opcodes():
        ref, pred = html.escape(reference[a0:a1]), html.escape(prediction[b0:b1])
        if op == "equal":
            out.append(ref)
        if op in ("delete", "replace"):
            out.append(f'<span style="background:#f7c5c5;text-decoration:line-through">{ref}</span>')
        if op in ("insert", "replace"):
            out.append(f'<span style="background:#c8f7c5">{pred}</span>')
    return "".join(out).replace("\n", "<br>")

box = "font-family:monospace;line-height:1.6;border:1px solid #ccc;padding:8px"

scores = pd.read_csv(os.path.join(run_dir, "dev_scores.csv"), keep_default_na=False)
print(scores.groupby("category")["cer"].describe()[["count", "mean", "min", "max"]].round(3).to_string())
PAGE = scores.sort_values("cer").iloc[-1]["page_id"]   # or type a page name, e.g. "dominy_018_p002"
row = scores.set_index("page_id").loc[PAGE]
from PIL import Image
img = Image.open(os.path.join(DATA, "images", PAGE + ".jpg"))
img.thumbnail((700, 900))
display(img)
display(HTML(f"<h4>{PAGE}: CER {row['cer']:.3f}</h4><div style='{box}'>"
             f"{coloured_diff(row['reference'], row['prediction'])}</div>"))

## Step 8. Save the run to S3 (recommended)

The adapter is about 120 MB; the big checkpoint files are left out.

In [ ]:
MY_BUCKET = "your-bucket-name"   # <-- your own bucket

!aws s3 sync {run_dir} s3://{MY_BUCKET}/churro-work/runs/{run_name} --exclude "checkpoints/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/churro-work/runs/{run_name}")

**Next:** change one setting and run Steps 4 to 8 again, or go to **Notebook 3** to read the
test pages with your best run. First shut this notebook's kernel down (Kernel → Shut Down Kernel)
so the GPU is free. **When you stop for the day, stop the instance.**